# Lanczos

给定厄密算符 $ H $ 计算它的基态

In [23]:
import quante as qt
import numpy as np

## 幂法

幂法：
$$
    H^{\Lambda} \left| \Psi  \right> \approx \left| \phi_{\text{ground}}  \right>
$$
当增加 ${ \Lambda }$，左边将接近右边

In [24]:
dim = 4000

# 生成随机的厄密矩阵
H = qt.generate.matrix.random_matrix(dim, mtype='GUE') - 1 * np.eye(dim)
np.allclose(H, H.conj().T)

True

In [25]:
res = np.linalg.eigvalsh(H)
res[0]

np.float64(-127.41523722082941)

但实际需要的不是每个矩阵元，只要知道矩阵怎么作用到一个向量上就行了

In [26]:
# 获取矩阵作用到向量上的方法：
matvec = H.dot

v = np.random.randn(dim) + 1j * np.random.randn(dim)
v /= np.linalg.norm(v)

np.allclose(matvec(v), H @ v)

True

利用 matvec 反复作用就能得到基态

In [27]:
Hv = v
for i in range(20):
    Hv = matvec(Hv)

# 可以近似基态能量
np.linalg.norm(Hv)**(1/20)  # exact: -126.94309301556632

np.float64(111.89092849573052)

增加 ${ \Gamma }$ 

In [28]:
iter_num = 100

Hv = v
for i in range(iter_num):
    Hv = matvec(Hv)
    Hv /= np.linalg.norm(Hv)

# # 可以近似基态能量
np.linalg.norm(matvec(Hv))  # exact: -126.94309301556632

np.float64(126.33868156167918)

## Krylov 方法

为减少迭代的次数，构建由下面这组基矢张成的空间：
$$
    \left| \psi  \right>,  H \left| \psi  \right>,  H^{2} \left| \psi  \right>, \cdots , H^{\Lambda} \left| \psi  \right>
$$
但是这一组基矢不正交，不能构建矩阵，所以需要正交化

最简单的正交化方法是 QR 分解

In [29]:
bases = [v]
iter_num = 20
for i in range(iter_num-1):
    bases.append(matvec(bases[-1]))

# 现在需要正交化 bases 中的基矢
bases = np.array(bases).T
bases.shape

(4000, 20)

In [30]:
res = np.linalg.qr(bases)
res.Q.shape

(4000, 20)

证明 QR 就是正交化手续，要验证：
- Q 矩阵每列彼此正交
- bases 中的任何一个基矢能由 Q 中的各列叠加得到

In [31]:
# 验证正交性：
for i in range(res.Q.shape[0]):
    for j in range(i+1, res.Q.shape[1]):
        assert np.allclose(res.Q[:, i].conj() @ res.Q[:, j], 0)

# 验证可以叠加出 bases 中的任何一个基矢：
for i in range(bases.shape[1]):
    rebuilt_base = sum(res.R[j, i] * res.Q[:, j] for j in range(res.Q.shape[1]))
                # \sum_j     c_ij     newvec_j   == bases[i]
    assert np.allclose(rebuilt_base, bases[:, i])

有了新的基矢：
$$
    \left| \phi_1  \right>,  \left| \phi_2  \right>, \cdots , \left| \phi_{\Lambda}  \right>
$$
之后可以在这个子空间中写出厄密算符的矩阵（厄密算符在 Krylov 空间中的投影）
$$
    h_{ij} = \langle \phi_{i} | H | \phi_{j} \rangle
$$

In [32]:
# 填写矩阵元的方法：
h = np.zeros((iter_num, iter_num), dtype=complex)

for i in range(iter_num):
    for j in range(iter_num):
        h[i, j] = res.Q[:, i].conj() @ matvec(res.Q[:, j])

# 或者直接通过矩阵方法来得到
h_matmul = res.Q.conj().T @ matvec(res.Q)

np.allclose(h_matmul, h)

True

通过对角化这个矩阵，就可以得到基态能量更好的近似

In [33]:
res = np.linalg.eigh(h)
res.eigenvalues[0]  # 只乘了 20 次，精确到小数点后两位
# exact: -126.94309301556632

np.float64(-125.90228255411988)

QR 还是慢了，可以进一步加快运算：
通过：
$$
    \left| \psi_{m + 1}  \right> = H \left| \psi_{m}  \right> - a_{m} \left| \psi_{m}  \right> - b_{m - 1} \left| \psi_{m - 1}  \right>
$$
的方法构建基矢：
$$
    \left| \psi_{1}  \right>, \left| \psi_{2}  \right>, \cdots , \left| \psi_{\Lambda} \right>
$$
其中：
$$
    a_{m} = \langle \psi_{m} | H | \psi_{m} \rangle/\langle \psi_{m} | \psi_{m} \rangle ,\quad\;\; b_{m - 1} = \langle \psi_{m} | \psi_{m} \rangle/\langle \psi_{m - 1} | \psi_{m - 1} \rangle
$$
并且前两个基矢为：${ \left| \psi_2  \right> = H \left| \psi_{1}  \right> - a_1 \left| \psi_{1}  \right> }$ 

In [34]:
newbases = []

v1 = v/np.linalg.norm(v)
newbases.append(v1)

a1 = v1.conj().T @ matvec(v1)
v2 = matvec(v1) - a1 * v1
newbases.append(v2)

for i in range(2, iter_num):
    a2, b1 = (v2.conj().T @ matvec(v2)) / (v2.conj().T @ v2), (v2.conj().T @ v2)/(v1.conj().T @ v1)
    
    v3 = matvec(v2) - a2 * v2 - b1 * v1
    newbases.append(v3)
    v2, v1 = v3, v2

newbases = [i/np.linalg.norm(i) for i in newbases] # 归一化

In [35]:
# 验证正交性：
for i in range(len(newbases)):
    for j in range(i+1, len(newbases)):
        assert np.allclose(newbases[i].conj() @ newbases[j], 0)

# 因为基矢是由 H^n |psi> 叠加出的，所以必然在 Krylov 空间中：

In [36]:
newbases = np.array(newbases).T
newbases.shape

(4000, 20)

In [37]:
h = newbases.conj().T @ matvec(newbases)
res = np.linalg.eigh(h)
res.eigenvalues[0]  # 结果一样

np.float64(-125.90228255411772)

这时 h 是三对角的实矩阵：

In [38]:
h

array([[-1.09606638-0.j, 62.93576337+0.j,  0.        +0.j,  0.        -0.j,  0.        -0.j, -0.        -0.j,  0.        -0.j, -0.        +0.j, -0.        -0.j, -0.        +0.j, -0.        -0.j, -0.        +0.j, -0.        +0.j, -0.        -0.j, -0.        -0.j, -0.        -0.j, -0.        -0.j, -0.        -0.j, -0.        +0.j, -0.        -0.j],
       [62.93576337-0.j, -0.14811558-0.j, 63.43923394-0.j,  0.        +0.j,  0.        -0.j,  0.        -0.j, -0.        -0.j, -0.        -0.j, -0.        +0.j, -0.        -0.j, -0.        +0.j, -0.        +0.j, -0.        +0.j, -0.        +0.j, -0.        -0.j, -0.        -0.j, -0.        -0.j, -0.        -0.j, -0.        -0.j, -0.        -0.j],
       [ 0.        -0.j, 63.43923394+0.j, -1.2324464 -0.j, 63.6687193 -0.j,  0.        -0.j, -0.        -0.j,  0.        -0.j, -0.        -0.j, -0.        -0.j, -0.        +0.j, -0.        +0.j, -0.        +0.j, -0.        +0.j, -0.        +0.j, -0.        -0.j, -0.        -0.j, -0.        -0.j, -0.  

实际上可以直接生成这个矩阵：
$$
\begin{align*}
    \langle \psi_{m - 1} | H | \psi_{m} \rangle &= \sqrt{b_{m - 1}}\\
    \langle \psi_{m} | H | \psi_{m} \rangle &= a_m \\
    \langle \psi_{m + 1} | H | \psi_{m} \rangle &= \sqrt{b_{m}}\\
\end{align*}
$$

In [39]:
h = np.zeros((iter_num, iter_num), dtype=float)

v /= np.linalg.norm(v)
v1 = v

a1 = np.real(v.conj().T @ matvec(v))
h[0, 0] = a1

v2 = matvec(v) - a1 * v
b1 = np.real(v2.conj().T @ v2)
h[0, 1] = h[1, 0] = np.sqrt(b1)

for i in range(1, iter_num-1):
    Hv2 = matvec(v2)
    
    a2 = np.real(v2.conj().T @ Hv2  / (v2.conj().T @ v2))
    h[i, i] = a2
    
    v3 = Hv2 - a2 * v2 - b1 * v1 
    b2 = np.real(v3.conj().T @ v3 / (v2.conj().T @ v2))
    h[i, i+1] = h[i+1, i] = np.sqrt(b2)
    
    v2, v1 = v3, v2
    b1 = b2

Hv2 = matvec(v2)
a2 = np.real(v2.conj().T @ Hv2  / (v2.conj().T @ v2))
h[i+1, i+1] = a2

h  # 得到了完全一样的矩阵 （一共只做了 20 次矩阵向量乘法）

array([[-1.09606638, 62.93576337,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [62.93576337, -0.14811558, 63.43923394,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        , 63.43923394, -1.2324464 , 63.6687193 ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , 63.6687193 , -0.4744531 , 64.2368235 ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.

In [40]:
res = np.linalg.eigh(h)
res.eigenvalues[0]  # 得到相同的结果

np.float64(-125.90228255411805)

## Chebyshev


In [41]:
import numpy as np
import quante as qt
from scipy.special import jv, iv

### **方法步骤**
1. **哈密顿量归一化**  
    首先药估计哈密顿量 $H$ 的谱范围，确定其最大和最小本征值 $E_{\text{max}}$ 和 $E_{\text{min}}$。

    定义：$a = \frac{E_{\text{max}} + E_{\text{min}}}{2}$，$b = \frac{E_{\text{max}} - E_{\text{min}}}{2}$

    切比雪夫多项式在区间 $[-1, 1]$ 上定义，需将哈密顿量 $H$ 缩放到此区间：
    $$
    \tilde{H} = \frac{H - a \cdot I}{b}
    $$

2. **切比雪夫多项式展开时间演化算符**  
    时间演化算符可展开为：
    $$
    e^{-iHt} \approx e^{-i a t} \sum_{n=0}^{N} c_n(t) T_n(\tilde{H}),
    $$
    其中 $T_n(x)$ 是第 $n$ 阶切比雪夫多项式，系数 $c_n(t) = (2 - \delta_{n0}) (-i)^n J_n(b t)$，$J_n$ 是贝塞尔函数。

3. **递推计算量子态演化**  
   利用 Clenshaw 递推法高效求和，避免直接计算高阶多项式：
   $$
   |\psi(t)\rangle = e^{-i a t} \sum_{n=0}^{N} c_n(t) |\phi_n\rangle,
   $$
   其中 $|\phi_n\rangle = T_n(\tilde{H}) |\psi(0)\rangle$ 通过递推关系 $|\phi_{n+1}\rangle = 2\tilde{H} |\phi_n\rangle - |\phi_{n-1}\rangle$ 生成。

4. **误差控制**  
   截断阶数 $N$ 由贝塞尔函数衰减特性决定，通常取 $N \propto b t + \log(\epsilon^{-1})$，$\epsilon$ 为允许误差。

---

### **优缺点**
- **优点**：对稀疏矩阵高效；长时间演化稳定性好；无需显式存储矩阵。
- **缺点**：需估计哈密顿量谱范围；短时间演化可能不如其他方法（如龙格-库塔）高效。

---

### **参考资料**
- Tal-Ezer, H., & Kosloff, R. (1984). ["An accurate and efficient scheme for propagating the time dependent Schrödinger equation"](https://doi.org/10.1063/1.447160). *The Journal of Chemical Physics*.  
- Leforestier, C., et al. (1991). ["A comparison of different propagation schemes for the time dependent Schrödinger equation"](https://doi.org/10.1063/1.460828). *Journal of Computational Physics*.
- https://github.com/Phyzch/Chebyshev_method


In [42]:
# 计算 exp(-1jH) 来验证算法
L = 5
t = 1.
N = 10
mat = qt.generate.matrix.heisenberg_matrix(L=L)

U = qt.linalg.expm(mat, c=-t*1j)

engs = np.linalg.eigvalsh(mat)
min_eng, max_eng = np.min(engs), np.max(engs)

a = (max_eng + min_eng) / 2
b = (max_eng - min_eng) / 2
I = np.eye(mat.shape[0])
omega = (mat - a * I)/b

coefs = [(1 if k == 0 else 2) * (-1j)**k * jv(k, b*t) for k in range(N)]

Tmat = [I, omega]
for k in range(2,N):
    Tmat.append(2 * (omega @ Tmat[k-1]) - Tmat[k-2])

print(np.linalg.norm(U - np.exp(-1j*a*t) * sum(coefs[k] * Tmat[k] for k in range(2))))
print(np.linalg.norm(U - np.exp(-1j*a*t) * sum(coefs[k] * Tmat[k] for k in range(3))))
print(np.linalg.norm(U - np.exp(-1j*a*t) * sum(coefs[k] * Tmat[k] for k in range(N))))

2.0436121621656853
0.5083625398348821
8.96434205777868e-08


In [43]:
# 计算 exp(-1jH)|psi> 来验证算法
L = 5
t = 1.
N = 10
mat = qt.generate.matrix.heisenberg_matrix(L=L)
initstate = np.random.randn(mat.shape[0])
initstate /= np.linalg.norm(initstate)

U = qt.linalg.expm(mat, c=-t*1j)
finalstate_exa = U @ initstate

engs = np.linalg.eigvalsh(mat)
min_eng, max_eng = np.min(engs), np.max(engs)

a = (max_eng + min_eng) / 2
b = (max_eng - min_eng) / 2
I = np.eye(mat.shape[0])
omega = (mat - a * I)/b

coefs = [(1 if k == 0 else 2) * (-1j)**k * jv(k, b*t) for k in range(N)]

tmp_state0 = initstate.copy()
tmp_state1 = (mat @ initstate - a * initstate)/b
finalstate_cheb = coefs[0] * tmp_state0 * np.exp(-1j*a*t)
finalstate_cheb += coefs[1] * tmp_state1 * np.exp(-1j*a*t)
for k in range(2,N):
    tmp_state0 = (2/b) * (mat @ tmp_state1 - a * tmp_state1) - tmp_state0
    tmp_state1, tmp_state0 = tmp_state0, tmp_state1
    finalstate_cheb += coefs[k] * tmp_state1 * np.exp(-1j*a*t)

print(np.linalg.norm(finalstate_exa - finalstate_cheb))
np.linalg.norm(finalstate_exa), np.linalg.norm(finalstate_cheb)

1.5335687836168342e-08


(np.float64(1.0000000000000002), np.float64(0.9999999960184742))

In [44]:
# 写成函数的形式
def chebyshev_evolve(mat:np.ndarray, initstate:np.ndarray, t:float, max_eng:float, min_eng:float, N:int) -> np.ndarray:
    """ Chebyshev evolution of a state under a Hamiltonian, `exp( - 1j H t) |initstate>`.
    This function uses Chebyshev polynomial expansion to evolve the state under the Hamiltonian mat.

    Parameters
    ----------
    mat : np.ndarray
        the Hamiltonian matrix
    initstate : np.ndarray
        the initial state vector
    t : float
        the time parameter for evolution
    max_eng : float
        maximum energy eigenvalue of the Hamiltonian
    min_eng : float
        minimum energy eigenvalue of the Hamiltonian
    N : int
        the number of Chebyshev polynomials to use

    Returns
    -------
    np.ndarray
        the final state vector after evolution
    
    Notes
    -----
    这是一个 Chebyshev 的原理验证函数。
    如果需要加速，可以考虑将 mat @ xxx 改为使用 gpu torch 来加速。
    对于更大规模的计算，需要考虑使用 petsc，相关的 c++ 程序见 https://github.com/Phyzch/Chebyshev_method
    
    Example
    -------
    >>> L, t, N = 5, 1., 10
    >>> mat = qt.generate.matrix.heisenberg_matrix(L=L)
    >>> initstate = np.random.randn(mat.shape[0])
    >>> initstate /= np.linalg.norm(initstate)
    >>> max_eng, min_eng = np.max(np.linalg.eigvalsh(mat)), np.min(np.linalg.eigvalsh(mat))
    >>> finalstate = chebyshev_evolve(mat, initstate, t, max_eng, min_eng, N)
    >>> np.linalg.norm(finalstate - qt.linalg.expm(mat, c=-t*1j) @ initstate)
    np.float64(1.5768894460867202e-08)
    """
    a = (max_eng + min_eng) / 2
    b = (max_eng - min_eng) / 2
    tmp_state0 = initstate.copy()
    tmp_state1 = (mat @ initstate - a * initstate)/b  #!! main time
    finalstate_cheb = jv(0, b*t) * tmp_state0 * np.exp(-1j*a*t)
    finalstate_cheb += 2 * (-1j) * jv(1, b*t) * tmp_state1 * np.exp(-1j*a*t)
    for k in range(2,N):
        tmp_state0 = (2/b) * (mat @ tmp_state1 - a * tmp_state1) - tmp_state0  #!! main time
        tmp_state1, tmp_state0 = tmp_state0, tmp_state1
        finalstate_cheb += 2 * (-1j)**k * jv(k, b*t) * tmp_state1 * np.exp(-1j*a*t)
    return finalstate_cheb

L, t, N = 5, 1., 10
mat = qt.generate.matrix.heisenberg_matrix(L=L)
initstate = np.random.randn(mat.shape[0])
initstate /= np.linalg.norm(initstate)
max_eng, min_eng = np.max(np.linalg.eigvalsh(mat)), np.min(np.linalg.eigvalsh(mat))
finalstate = chebyshev_evolve(mat, initstate, t, max_eng, min_eng, N)
np.linalg.norm(finalstate - qt.linalg.expm(mat, c=-t*1j) @ initstate)

np.float64(1.3722505517475513e-08)

## Krylov Example

In [12]:
import quante as qt
mat = qt.generate.matrix.heisenberg_matrix(L=10, sparse=True)
x0 = qt.generate.state.random(mat.shape[0])
val, vec , _ = qt.linalg.krylov.eigsolve(
    mat, x0, howmany=1, which='SR', isherm=True
)
val

running Lanczos ...


array([-4.25803521, -3.93067359])

In [10]:
import quante as qt
mat = qt.generate.matrix.heisenberg_matrix(L=20, sparse=True)
x0 = qt.generate.state.random(mat.shape[0]).real

from scipy.sparse.linalg import eigsh
with qt.basicfun.Timer('scipy'):
    val, vec = eigsh(
        mat, v0=x0, k=1, which='SA'
    )
print(val)

with qt.basicfun.Timer("quante (numpy)"):
    val, vec , _ = qt.linalg.krylov.eigsolve(
        mat, x0, howmany=1, which='SR', isherm=True
    )
print(val)

from quante.bridge.torch_utils import totc
mat, x0 = totc(mat), totc(x0)
with qt.basicfun.Timer('quante (torch-cpu)'):
    val, vec , _ = qt.linalg.krylov.eigsolve(
        mat, x0, howmany=1, which='SR', isherm=True
    )
print(val)

import torch as tc
if tc.cuda.is_available():
    mat, x0 = mat.to('cuda'), x0.to('cuda')
    with qt.basicfun.Timer('quante (torch-gpu)'):
        val, vec , _ = qt.linalg.krylov.eigsolve(
            mat, x0, howmany=1, which='SR', isherm=True
        )
    print(val)

scipy: 2.5976510000182316 seconds


[-8.68247333]
running Lanczos ...


quante (numpy): 1.7192967000009958 seconds


[-8.68247333]
running Lanczos ...


quante (torch-cpu): 1.3955421000136994 seconds


[-8.68247333]
